In [2]:
import fastapi
import uvicorn
from pathlib import Path
import pandas as pd
from fastapi import FastAPI, HTTPException

In [3]:
PROJECT_DIR = Path(
    r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot"
)

OUTPUT_DIR = PROJECT_DIR / "Output"

RESULTS_PATH = OUTPUT_DIR / "reconciliation_results.csv"


# --------------------------------------------------
# FastAPI application
# --------------------------------------------------

app = FastAPI(
    title="ReconPilot API",
    description="AI-powered financial reconciliation API",
    version="0.1.0",
)


# --------------------------------------------------
# Data loader
# --------------------------------------------------

def load_results():
    if not RESULTS_PATH.exists():
        raise FileNotFoundError(
            f"Reconciliation results not found: {RESULTS_PATH}"
        )

    return pd.read_csv(RESULTS_PATH)


# --------------------------------------------------
# Health check
# --------------------------------------------------

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "service": "reconpilot",
    }


# --------------------------------------------------
# Reconciliation summary
# --------------------------------------------------

@app.get("/reconciliation/summary")
def reconciliation_summary():

    results = load_results()

    total_records = len(results)

    auto_reconciled = int(
        (results["status"] == "AUTO_RECONCILED").sum()
    )

    exceptions = int(
        (results["status"] == "EXCEPTION").sum()
    )

    exception_rate = (
        exceptions / total_records
        if total_records > 0
        else 0
    )

    return {
        "total_records": total_records,
        "auto_reconciled": auto_reconciled,
        "exceptions": exceptions,
        "exception_rate": round(exception_rate, 4),
    }


# --------------------------------------------------
# Get individual reconciliation case
# --------------------------------------------------

@app.get("/reconciliation/{payment_id}")
def get_reconciliation(payment_id: str):

    results = load_results()

    match = results[
        results["payment_id"] == payment_id
    ]

    if match.empty:
        raise HTTPException(
            status_code=404,
            detail=f"Payment {payment_id} not found",
        )

    record = match.iloc[0].to_dict()

    # Convert NaN values to None
    record = {
        key: (
            None
            if pd.isna(value)
            else value
        )
        for key, value in record.items()
    }

    return record